## Mineru 服务测试

输入：`data/raw/e08/pdf/*.pdf`。输出：`_scratch/ocr/<stem>.md` —— 探索产物，
**不是** `data/processed/` 的流水线工件。

In [ ]:
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "AGENTS.md").exists():          # notebook 可能从任意目录启动
    REPO = REPO.parent

from e08.mineru_ocr import aocr_pdf

PDF_DIR = REPO / "data/raw/e08/pdf"
OUT_DIR = Path.cwd() / "_scratch/ocr"             # 探索产物，不进 data/processed

pdf = PDF_DIR / "2025-Zep.pdf"
assert pdf.is_file(), pdf

### 跑一篇

单独一个 cell：一篇 20 页上下的论文要几十秒到几分钟，下面的落盘 cell 反复改时不必重跑。
同步版 `ocr_pdf` 在 Jupyter 里会直接抛 `RuntimeError`（已有事件循环），所以用 `await`。

In [ ]:
md_text = await aocr_pdf(pdf)
print(f"{len(md_text)} chars")
print(md_text[:800])

### 落盘

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
out = OUT_DIR / f"{pdf.stem}.md"
out.write_text(md_text, encoding="utf-8")
print(out.relative_to(REPO), out.stat().st_size, "bytes")